In [ ]:
!pip install segmentation-models-pytorch
!pip install albumentations

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import os
import json
import glob
import base64
import random
from io import BytesIO

import cv2
import numpy as np
from PIL import Image
from scipy import ndimage
from scipy.ndimage import gaussian_filter

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms, models

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from sklearn.metrics import (f1_score, recall_score, confusion_matrix,
                             precision_score, balanced_accuracy_score, roc_curve)
from sklearn.model_selection import StratifiedKFold, KFold

import albumentations as A
import segmentation_models_pytorch as smp

from torch.amp import autocast, GradScaler

In [ ]:
# ==========================================
# 1. GLOBAL HYPERPARAMETERS
# ==========================================
CURRENT_CROP_FACTOR = 0.50     # ablation "w/o crop": 1.0
CURRENT_CLAHE_LIMIT = 3.0      # ablation "w/o CLAHE": 0

# Preprocessing ablation switches: True = step enabled, False = step disabled
USE_PERCENTILE    = True       # percentile normalization (2-98)
USE_BILATERAL     = True       # bilateral filter
USE_IMAGENET_NORM = True       # ImageNet mean/std normalization
IMG_SIZE = 300
FALLBACK_MIN_PIXELS = int(IMG_SIZE * IMG_SIZE * 0.01)

# ==========================================
# 2. DATASET PATHS
# ==========================================
# Folder containing "dataset_pulito" (update this path if needed)
BASE_DATASET = "/kaggle/input/datasets/flavioberardocco/dataset-pulito/dataset_pulito"

# Training masks: the U-Net is trained ONLY on the training-set masks
MASKS_DIR      = os.path.join(BASE_DATASET, "maschere_train")
# Test masks: used only to compute Dice/IoU on the test set
MASKS_TEST_DIR = os.path.join(BASE_DATASET, "maschere_test")

# Train/test split folders
COMBINED_DATA_DIR = os.path.join(BASE_DATASET, "train")
TEST_DIR          = os.path.join(BASE_DATASET, "test")

# Path check
print("--- PATH CHECK ---")
for name, p in [("MASKS", MASKS_DIR), ("TEST MASKS", MASKS_TEST_DIR), ("TRAIN", COMBINED_DATA_DIR), ("TEST", TEST_DIR)]:
    if os.path.exists(p):
        print(f"✅ {name} FOUND")
    else:
        print(f"❌ {name} NOT FOUND")

# ==========================================
# 3. ARCHITECTURES AND TRAINING
# ==========================================
MIN_F1_FOR_ENSEMBLE = 0.70
MIN_MODELS_FALLBACK = 3

BACKBONES_UNET    = ['densenet121', 'resnet18', 'efficientnet-b0']
ARCHITECTURES_CLS = ['densenet121', 'resnet18', 'efficientnet_v2_s']

seg_results_table = []

LABEL_SMOOTH, FREEZE_EPOCHS = 0.05, 3
HEAD_LR, BACKBONE_LR = 5e-4, 1e-5
ES_PATIENCE = 15
SEEDS = [42, 123, 7]

# ==========================================
# 4. DEVICE SETUP
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pos_weight_tensor = torch.tensor([1.5]).to(device)

print(f"\n🚀 Device: {device}")

In [ ]:
class AddGaussianNoise(object):
    """Adds Gaussian noise with probability p."""
    def __init__(self, mean=0., std=0.03, p=0.4):
        self.std = std
        self.mean = mean
        self.p = p

    def __call__(self, tensor):
        if torch.rand(1).item() < self.p:
            noise = torch.randn_like(tensor) * self.std + self.mean
            return tensor + noise
        return tensor

# CLASSIFICATION DATASET
class DatasetWrapper(Dataset):
    """Reads the images from the cache (already preprocessed)."""
    def __init__(self, subset, transform=None, image_cache=None):
        self.subset = subset
        self.transform = transform
        self.image_cache = image_cache

    def __getitem__(self, index):
        img_path, label = self.subset.dataset.samples[self.subset.indices[index]]
        if self.image_cache is not None and img_path in self.image_cache:
            img = self.image_cache[img_path]
        else:
            img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

    def __len__(self):
        return len(self.subset)

def smooth_labels(targets, smoothing=0.05):
    return targets * (1.0 - smoothing) + 0.5 * smoothing

# FOCAL LOSS
class FocalBCEWithLogitsLoss(nn.Module):
    """
    Focal Loss for binary classification with logits.
    Down-weights easy samples and focuses on hard ones:
    well-classified samples contribute less to the loss.
    """
    def __init__(self, gamma=2.0, pos_weight=None):
        super().__init__()
        self.gamma = gamma
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(
            logits, targets,
            pos_weight=self.pos_weight,
            reduction='none'
        )
        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        focal_weight = (1 - p_t) ** self.gamma
        loss = focal_weight * bce
        return loss.mean()


# TTA for validation during training (classification)
@torch.no_grad()
def classify_with_tta_light(model, images, device):
    """Light TTA (horizontal flip only) for validation during training."""
    images = images.to(device)
    with autocast(device_type='cuda'):
        logit_orig  = model(images)
        logit_hflip = model(torch.flip(images, dims=[-1]))
    return (logit_orig + logit_hflip) / 2.0


# TTA for the final test (classification)
@torch.no_grad()
def classify_with_tta(model, images, device):
    """Full TTA: original + hflip + 2 scales + hflip of the upscaled image."""
    images = images.to(device)
    B, C, H, W = images.shape

    with autocast(device_type='cuda'):
        logit_orig = model(images)
        logit_hflip = model(torch.flip(images, dims=[-1]))

        scaled_up = F.interpolate(images, scale_factor=1.1,
                                  mode='bilinear', align_corners=False)
        dh = (scaled_up.shape[2] - H) // 2
        dw = (scaled_up.shape[3] - W) // 2
        scaled_up = scaled_up[:, :, dh:dh+H, dw:dw+W]
        logit_sup = model(scaled_up)

        scaled_dn = F.interpolate(images, scale_factor=0.9,
                                  mode='bilinear', align_corners=False)
        pad_h = H - scaled_dn.shape[2]
        pad_w = W - scaled_dn.shape[3]
        scaled_dn = F.pad(scaled_dn,
                          [pad_w//2, pad_w - pad_w//2,
                           pad_h//2, pad_h - pad_h//2])
        logit_sdn = model(scaled_dn)

        logit_hf_sup = model(torch.flip(scaled_up, dims=[-1]))

    return (logit_orig + logit_hflip + logit_sup + logit_sdn + logit_hf_sup) / 5.0

In [ ]:
# SEGMENTATION AUGMENTATION
def get_seg_train_augmentation(img_size=IMG_SIZE):
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.Affine(
            translate_percent=0.5,
            scale=(0.9, 1.1),
            rotate=(-10, 10),
            border_mode=cv2.BORDER_CONSTANT,
            p=0.6,
        ),
        A.ElasticTransform(
            alpha=40, sigma=40 * 0.05,
            p=0.3
        ),
        A.GridDistortion(
            num_steps=5, distort_limit=0.15,
            p=0.2
        ),
        A.RandomBrightnessContrast(
            brightness_limit=0.15, contrast_limit=0.15,
            p=0.5
        ),
        A.GaussNoise(
            std_range=(0.02, 0.1),
            p=0.3
        ),
        A.GaussianBlur(
            blur_limit=(3, 5),
            p=0.2
        ),
    ])

In [ ]:
# SEGMENTATION DATASET
class SegmentationDatasetV2(Dataset):
    IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
    IMAGENET_STD  = np.array([0.229, 0.224, 0.225])

    def __init__(self, json_paths, img_size=IMG_SIZE, augmentation=None):
        self.json_paths   = json_paths
        self.img_size     = img_size
        self.augmentation = augmentation

    def __len__(self):
        return len(self.json_paths)

    def _load_and_preprocess(self, json_path):
        with open(json_path, 'r') as f:
            data = json.load(f)

        img_pil = Image.open(
            BytesIO(base64.b64decode(data['imageData']))
        ).convert('RGB')
        img_np = np.array(img_pil)

        orig_h = data['imageHeight']
        orig_w = data['imageWidth']

        mask_np = np.zeros((orig_h, orig_w), dtype=np.uint8)
        for shape in data['shapes']:
            if shape['label'].lower() == 'mps' and shape['shape_type'] == 'polygon':
                pts = np.array(shape['points'], dtype=np.int32).reshape(-1, 1, 2)
                cv2.fillPoly(mask_np, [pts], color=1)

        # PREPROCESSING
        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)

        # Percentile normalization
        if USE_PERCENTILE:
            p2, p98 = np.percentile(gray, (2, 98))
            if p98 > p2:
                gray = np.clip((gray - p2) / (p98 - p2) * 255, 0, 255).astype(np.uint8)

        # Bilateral filter + CLAHE
        if USE_BILATERAL:
            gray = cv2.bilateralFilter(gray, 5, 75, 75)
        if CURRENT_CLAHE_LIMIT > 0:
            clahe = cv2.createCLAHE(clipLimit=CURRENT_CLAHE_LIMIT, tileGridSize=(8, 8))
            gray = clahe.apply(gray)

        # Central crop
        h, w = gray.shape
        new_width = int(w * CURRENT_CROP_FACTOR)
        start_x = (w - new_width) // 2
        gray_crop = gray[:, start_x:start_x + new_width]
        mask_crop = mask_np[:, start_x:start_x + new_width]

        # Resize
        img_resized = cv2.resize(gray_crop, (self.img_size, self.img_size),
                                  interpolation=cv2.INTER_CUBIC)
        mask_resized = cv2.resize(mask_crop, (self.img_size, self.img_size),
                                   interpolation=cv2.INTER_NEAREST)

        img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_GRAY2RGB)
        return img_rgb, mask_resized

    def __getitem__(self, idx):
        img_np, mask_np = self._load_and_preprocess(self.json_paths[idx])

        # Augmentation
        if self.augmentation is not None:
            augmented = self.augmentation(image=img_np, mask=mask_np)
            img_np  = augmented['image']
            mask_np = augmented['mask']

        img_float = img_np.astype(np.float32) / 255.0
        if USE_IMAGENET_NORM:
            img_float = (img_float - self.IMAGENET_MEAN) / self.IMAGENET_STD
        img_tensor = torch.from_numpy(img_float).permute(2, 0, 1).float()

        mask_tensor = torch.from_numpy((mask_np > 0.5).astype(np.float32)).unsqueeze(0)
        return img_tensor, mask_tensor

In [ ]:
# SEGMENTATION LOSS FUNCTIONS

class BoundaryLoss(nn.Module):
    """
    Boundary Loss: weights the BCE so that
    pixels close to the mask border get a higher weight.
    """
    def __init__(self, theta0=3, theta=5):
        super().__init__()
        self.theta0 = theta0
        self.theta  = theta

    def forward(self, logits, targets):
        batch_size = targets.size(0)
        total_loss = 0.0

        for i in range(batch_size):
            target_np = targets[i, 0].cpu().numpy().astype(np.uint8)

            if target_np.sum() > 0 and target_np.sum() < target_np.size:
                dist_inside  = ndimage.distance_transform_edt(target_np)
                dist_outside = ndimage.distance_transform_edt(1 - target_np)
                dist_border  = dist_inside + dist_outside
                weight_map = 1.0 + self.theta0 * np.exp(-dist_border / self.theta)
            else:
                weight_map = np.ones_like(target_np, dtype=np.float32)

            weight_tensor = torch.from_numpy(weight_map).float().to(logits.device)
            bce_per_pixel = F.binary_cross_entropy_with_logits(
                logits[i, 0], targets[i, 0], reduction='none'
            )
            total_loss += (bce_per_pixel * weight_tensor).mean()

        return total_loss / batch_size


class CombinedSegLoss(nn.Module):
    """
    Combined loss:
      0.40 × Tversky Focal Loss (global overlap, focus on FN)
      0.35 × standard BCE
      0.25 × Boundary Loss (contour accuracy)
    """
    def __init__(self, alpha=0.7, beta=0.3, gamma=2.0, smooth=1e-6):
        super().__init__()
        self.alpha  = alpha
        self.beta   = beta
        self.gamma  = gamma
        self.smooth = smooth
        self.boundary_loss = BoundaryLoss(theta0=3, theta=5)

    def _tversky(self, pred, target):
        flat_pred   = pred.view(-1)
        flat_target = target.view(-1)
        TP = (flat_pred * flat_target).sum()
        FN = ((1 - flat_pred) * flat_target).sum()
        FP = (flat_pred * (1 - flat_target)).sum()
        return (TP + self.smooth) / (TP + self.alpha * FN + self.beta * FP + self.smooth)

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)

        # Tversky Focal Loss
        tversky_loss  = 1.0 - self._tversky(probs, targets)
        focal_tversky = tversky_loss ** (1.0 / self.gamma)

        # Standard BCE
        bce = F.binary_cross_entropy_with_logits(logits, targets)

        # Boundary Loss
        boundary = self.boundary_loss(logits, targets)

        return 0.40 * focal_tversky + 0.35 * bce + 0.25 * boundary

In [ ]:
# SEGMENTATION METRICS
def dice_coefficient(pred_mask, true_mask, threshold=0.5, smooth=1e-6):
    """Range [0,1], 1 = perfect overlap."""
    pred_bin     = (torch.sigmoid(pred_mask) > threshold).float()
    intersection = (pred_bin * true_mask.float()).sum()
    return ((2.0 * intersection + smooth) /
            (pred_bin.sum() + true_mask.float().sum() + smooth)).item()


def iou_score(pred_mask, true_mask, threshold=0.5, smooth=1e-6):
    """Stricter than Dice on false positives."""
    pred_bin     = (torch.sigmoid(pred_mask) > threshold).float()
    intersection = (pred_bin * true_mask.float()).sum()
    union        = pred_bin.sum() + true_mask.float().sum() - intersection
    return ((intersection + smooth) / (union + smooth)).item()

In [ ]:
# POST-PROCESSING OF THE PREDICTED MASK
def refine_mask(mask_np, min_area_ratio=0.005, closing_iter=2):
    """
    Post-processing of the predicted mask:
      1. Morphological closing: fills small holes in the suture
      2. Removal of components that are too small (noise)
    """
    if mask_np.sum() == 0:
        return mask_np

    mask_closed = ndimage.binary_closing(
        mask_np.astype(bool), iterations=closing_iter
    ).astype(np.float32)

    min_area = int(mask_np.size * min_area_ratio)
    labeled, n_components = ndimage.label(mask_closed)
    for i in range(1, n_components + 1):
        if (labeled == i).sum() < min_area:
            mask_closed[labeled == i] = 0

    return mask_closed

In [ ]:
# SEGMENTATION TTA WITH HORIZONTAL FLIP
def predict_with_tta(model, img_tensor, device):
    model.eval()
    img_tensor = img_tensor.to(device)

    with torch.no_grad():
        prob_orig = torch.sigmoid(model(img_tensor))

        img_flipped  = torch.flip(img_tensor, dims=[-1])
        prob_flipped = torch.sigmoid(model(img_flipped))
        prob_flipped = torch.flip(prob_flipped, dims=[-1])

        prob_avg = (prob_orig + prob_flipped) / 2.0

    return prob_avg

In [ ]:
# PREPROCESSING PIPELINE
class FinalPreprocessing:
    """
    Filters for false-positive U-Net masks:
      1. FALLBACK_MIN_PIXELS: if the mask is too small -> skip
      2. CONFIDENCE_THRESHOLD: if the mean confidence inside the segmented
         region is too low -> skip (the U-Net is not confident)
      3. MAX_MASK_RATIO: if the mask covers too much of the image
         -> probably a FP (the true suture region is small)

    Parameter use_segmentation:
      - True  = background attenuation via U-Net (complete pipeline)
      - False = skips the whole U-Net step (pipeline without segmentation)
    """

    def __init__(self, width_factor, clip_limit=None,
                 target_size=None,
                 unet_model=None, unet_device=None, use_tta=True,
                 blur_sigma=5.0, bg_attenuation=0.25,
                 confidence_threshold=0.65,
                 max_mask_ratio=0.35,
                 use_segmentation=True):
        self.width_factor   = width_factor
        self.clip_limit     = clip_limit
        self.target_size    = target_size if target_size else (IMG_SIZE, IMG_SIZE)
        self.unet_model     = unet_model
        self.unet_device    = unet_device
        self.use_tta        = use_tta
        self.blur_sigma     = blur_sigma
        self.bg_attenuation = bg_attenuation
        self.confidence_threshold = confidence_threshold
        self.max_mask_ratio       = max_mask_ratio
        self.use_segmentation     = use_segmentation
        self.normalize      = transforms.Normalize(
            mean=[0.485, 0.456, 0.406] if USE_IMAGENET_NORM else [0., 0., 0.],
            std=[0.229, 0.224, 0.225] if USE_IMAGENET_NORM else [1., 1., 1.]
        )

    def __call__(self, img_pil):
        img_full = np.array(img_pil.convert('L'))
        if img_full.size == 0:
            return img_pil.convert('RGB')

        # Percentile normalization
        p2, p98 = np.percentile(img_full, (2, 98))
        if USE_PERCENTILE and p98 > p2:
            img_norm = np.clip((img_full - p2) / (p98 - p2) * 255, 0, 255).astype(np.uint8)
        else:
            img_norm = img_full.astype(np.uint8)

        # Bilateral filter + CLAHE
        img_processed = cv2.bilateralFilter(img_norm, 5, 75, 75) if USE_BILATERAL else img_norm
        if self.clip_limit is not None and self.clip_limit > 0:
            clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=(8, 8))
            img_processed = clahe.apply(img_processed)

        # Central crop + resize
        h, w = img_processed.shape
        new_width = int(w * self.width_factor)
        start_x = (w - new_width) // 2
        img_crop = img_processed[:, start_x:start_x + new_width]
        img_rgb = Image.fromarray(
            cv2.resize(img_crop, self.target_size, interpolation=cv2.INTER_CUBIC)
        ).convert('RGB')

        # U-Net soft mask if use_segmentation=True
        if self.use_segmentation and self.unet_model is not None:
            img_tensor = self.normalize(transforms.ToTensor()(img_rgb)).unsqueeze(0)

            if self.use_tta:
                prob_tensor = predict_with_tta(
                    self.unet_model, img_tensor, self.unet_device
                )
                prob_map = prob_tensor.squeeze().cpu().numpy()
            else:
                img_tensor = img_tensor.to(self.unet_device)
                with torch.no_grad():
                    prob_map = torch.sigmoid(
                        self.unet_model(img_tensor)
                    ).squeeze().cpu().numpy()

            binary_mask = (prob_map > 0.5).astype(np.float32)
            mask_pixels = binary_mask.sum()
            total_pixels = binary_mask.size

            # FILTER 1: minimum size check
            if mask_pixels < FALLBACK_MIN_PIXELS:
                return img_rgb

            # FILTER 2: confidence check
            if mask_pixels > 0:
                mean_confidence = prob_map[binary_mask > 0.5].mean()
                if mean_confidence < self.confidence_threshold:
                    return img_rgb

            # FILTER 3: maximum size check
            mask_ratio = mask_pixels / total_pixels
            if mask_ratio > self.max_mask_ratio:
                return img_rgb

            # Post-processing
            binary_refined = refine_mask(binary_mask)

            prob_blurred = gaussian_filter(prob_map, sigma=self.blur_sigma)
            soft_mask = np.maximum(binary_refined * 0.8, prob_blurred * 0.7)
            soft_mask = np.clip(soft_mask, self.bg_attenuation, 1.0)

            img_np_out = np.array(img_rgb).astype(np.float32)
            img_rgb = Image.fromarray(
                (img_np_out * soft_mask[:, :, np.newaxis]).astype(np.uint8)
            )

        return img_rgb

In [ ]:
# U-Net with encoder chosen from {'densenet121', 'resnet18', 'efficientnet-b0'},
# pre-trained on ImageNet.
def get_unet_segmentation(backbone_name='efficientnet-b0', pretrained=True):
    return smp.Unet(
        encoder_name=backbone_name,
        encoder_weights='imagenet' if pretrained else None,
        in_channels=3,
        classes=1,
        activation=None
    )

In [ ]:
# U-NET TRAINING ON A SINGLE FOLD
def train_unet_single_fold(unet_model, train_loader, val_loader, device,
                            seg_epochs=80, es_patience=12, save_path='best_unet.pth'):

    criterion = CombinedSegLoss(alpha=0.7, beta=0.3, gamma=2.0)

    optimizer = optim.AdamW([
        {'params': unet_model.encoder.parameters(),           'lr': 1e-5},
        {'params': unet_model.decoder.parameters(),           'lr': 5e-4},
        {'params': unet_model.segmentation_head.parameters(), 'lr': 5e-4},
    ], weight_decay=0.01)

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5
    )

    best_val_dice = 0.0
    best_val_iou = 0.0
    es_counter = 0

    for epoch in range(seg_epochs):
        # Training phase
        unet_model.train()
        train_loss = 0.0
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            loss = criterion(unet_model(imgs), masks)
            loss.backward()
            nn.utils.clip_grad_norm_(unet_model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * imgs.size(0)

        # Validation phase
        unet_model.eval()
        val_loss, val_dice_list, val_iou_list = 0.0, [], []
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                logits = unet_model(imgs)
                val_loss += criterion(logits, masks).item() * imgs.size(0)
                val_dice_list.append(dice_coefficient(logits, masks))
                val_iou_list.append(iou_score(logits, masks))

        avg_train = train_loss / len(train_loader.dataset)
        avg_val   = val_loss / len(val_loader.dataset)
        avg_dice  = np.mean(val_dice_list)
        avg_iou   = np.mean(val_iou_list)

        scheduler.step(avg_dice)
        current_lr = optimizer.param_groups[0]['lr']

        print(f"  Ep {epoch+1:02d}/{seg_epochs} | "
              f"Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f} | "
              f"Dice: {avg_dice:.4f} | IoU: {avg_iou:.4f} | "
              f"LR: {current_lr:.0e} | ES: {es_counter}/{es_patience}")

        if avg_dice > best_val_dice:
            best_val_dice = avg_dice
            best_val_iou = avg_iou
            es_counter = 0
            torch.save(unet_model.state_dict(), save_path)
            print(f"    ✓ New best Dice: {best_val_dice:.4f} — saved.")
        else:
            es_counter += 1
            if es_counter >= es_patience:
                print(f"    Early stopping at epoch {epoch+1}.")
                break

    return best_val_dice, best_val_iou

# K-Fold Cross-Validation for segmentation
def train_unet_kfold(json_paths, device, k_folds=5, seg_epochs=80, es_patience=12,
                     backbone_name='efficientnet-b0',
                     checkpoint_prefix='best_unet'):
    """
    Trains k_folds U-Net models and saves one checkpoint per fold.
    """
    kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)
    fold_results = []
    checkpoint_paths = []

    print(f"\n{'═'*60}")
    print(f"  SEGMENTATION — {k_folds}-Fold Cross-Validation  [backbone: {backbone_name}]")
    print(f"  Total samples: {len(json_paths)}")
    print(f"{'═'*60}")

    for fold, (train_idx, val_idx) in enumerate(kf.split(json_paths)):
        print(f"\n{'─'*50}")
        print(f"  FOLD {fold+1}/{k_folds}")
        print(f"{'─'*50}")

        save_path = f'{checkpoint_prefix}_fold_{fold+1}.pth'
        checkpoint_paths.append(save_path)

        train_jsons = [json_paths[i] for i in train_idx]
        val_jsons   = [json_paths[i] for i in val_idx]
        print(f"  Train: {len(train_jsons)} | Val: {len(val_jsons)}")

        train_loader = DataLoader(
            SegmentationDatasetV2(train_jsons, augmentation=get_seg_train_augmentation()),
            batch_size=8, shuffle=True, num_workers=2, pin_memory=True
        )
        val_loader = DataLoader(
            SegmentationDatasetV2(val_jsons, augmentation=None),
            batch_size=8, shuffle=False, num_workers=2, pin_memory=True
        )

        model = get_unet_segmentation(backbone_name=backbone_name,
                                      pretrained=True).to(device)

        best_dice, best_iou = train_unet_single_fold(
            model, train_loader, val_loader, device,
            seg_epochs=seg_epochs, es_patience=es_patience,
            save_path=save_path
        )
        fold_results.append({'fold': fold + 1, 'best_dice': best_dice, 'best_iou': best_iou})

    dice_scores = [r['best_dice'] for r in fold_results]
    print(f"\n{'═'*60}")
    print(f"  SEGMENTATION K-FOLD RESULTS  [{backbone_name}]")
    print(f"{'═'*60}")
    print(f"  Mean Dice:  {np.mean(dice_scores):.4f} ± {np.std(dice_scores):.4f}")
    print(f"  Dice min:    {np.min(dice_scores):.4f}  |  Dice max: {np.max(dice_scores):.4f}")
    print(f"{'═'*60}")

    return fold_results, checkpoint_paths

In [ ]:
# U-NET ENSEMBLE
class UNetEnsemble(nn.Module):
    def __init__(self, checkpoint_paths, device, use_tta=True,
                 backbone_name='efficientnet-b0'):
        super().__init__()
        self.device  = device
        self.use_tta = use_tta
        self.models  = nn.ModuleList()

        for path in checkpoint_paths:
            model = get_unet_segmentation(backbone_name=backbone_name,
                                          pretrained=False)
            model.load_state_dict(
                torch.load(path, map_location=device, weights_only=True)
            )
            model.eval()
            self.models.append(model)

        self.to(device)

    def forward(self, x):
        all_probs = []
        for model in self.models:
            if self.use_tta:
                prob = predict_with_tta(model, x, self.device)
            else:
                with torch.no_grad():
                    prob = torch.sigmoid(model(x))
            all_probs.append(prob)
        prob_mean = torch.stack(all_probs).mean(dim=0)

        prob_clamped = prob_mean.clamp(1e-6, 1 - 1e-6)
        logits = torch.log(prob_clamped / (1 - prob_clamped))
        return logits

In [ ]:
# SEGMENTATION EVALUATION
def evaluate_segmentation_metrics(model, val_loader, device, backbone_name=None):
    """Computes per-image Dice and IoU on the given loader."""
    model.eval()
    dice_scores, iou_scores = [], []

    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)
            for i in range(imgs.size(0)):
                dice_scores.append(dice_coefficient(logits[i:i+1], masks[i:i+1]))
                iou_scores.append(iou_score(logits[i:i+1], masks[i:i+1]))

    print(f"\n{'═'*50}")
    print(f"  SEGMENTATION REPORT — ({len(dice_scores)} samples)")
    print(f"{'═'*50}")
    print(f"  Mean Dice:  {np.mean(dice_scores):.4f} ± {np.std(dice_scores):.4f}")
    print(f"  Mean IoU:   {np.mean(iou_scores):.4f}  ± {np.std(iou_scores):.4f}")
    print(f"  Dice min:    {np.min(dice_scores):.4f}  |  Dice max: {np.max(dice_scores):.4f}")
    print(f"{'═'*50}")

    plt.figure(figsize=(8, 4))
    plt.hist(dice_scores, bins=15, color='steelblue', edgecolor='white', alpha=0.85)
    plt.axvline(np.mean(dice_scores), color='red', linestyle='--',
                label=f'Mean: {np.mean(dice_scores):.3f}')
    plt.xlabel('Dice Coefficient')
    plt.ylabel('Frequency')
    plt.title('Dice distribution')
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Per-image Dice and IoU on the test set (for bootstrap confidence intervals)
    if backbone_name is not None:
        pd.DataFrame({'run': 'full', 'backbone': backbone_name, 'idx': range(len(dice_scores)),
                      'dice': dice_scores, 'iou': iou_scores}
                     ).to_csv(f'seg_test_{backbone_name}.csv', index=False)

    return np.mean(dice_scores), np.mean(iou_scores), np.std(dice_scores), np.std(iou_scores)

In [ ]:
# SEGMENTATION RUN

json_paths = sorted(glob.glob(os.path.join(MASKS_DIR, '*.json')))
json_paths = [p for p in json_paths if not os.path.basename(p).startswith('._')]
print(f"JSON found: {len(json_paths)}")

if len(json_paths) == 0:
    raise RuntimeError("No JSON found. Check MASKS_DIR.")

unet_ensembles = {}

# Final evaluation on the TEST masks (never seen by the U-Net)
test_json_paths = sorted(glob.glob(os.path.join(MASKS_TEST_DIR, '*.json')))
print(f"Test JSON: {len(test_json_paths)}")
full_seg_loader = DataLoader(
    SegmentationDatasetV2(test_json_paths, augmentation=None),
    batch_size=8, shuffle=False, num_workers=2, pin_memory=True
)

for backbone_name in BACKBONES_UNET:

    # Training 5-fold Cross Validation
    seg_fold_results, seg_checkpoint_paths = train_unet_kfold(
        json_paths, device, k_folds=5, seg_epochs=80, es_patience=12,
        backbone_name=backbone_name,
        checkpoint_prefix=f'best_unet_{backbone_name}'
    )

    # Ensemble
    ensemble = UNetEnsemble(
        seg_checkpoint_paths, device, use_tta=True,
        backbone_name=backbone_name
    )

    # Ensemble evaluation on the test masks
    dice_ensemble, iou_ensemble, dice_ens_std, iou_ens_std = evaluate_segmentation_metrics(
        ensemble, full_seg_loader, device, backbone_name=backbone_name
    )

    # Store metrics
    fold_dices = [r['best_dice'] for r in seg_fold_results]
    fold_ious = [r['best_iou'] for r in seg_fold_results]
    seg_results_table.append({
        'backbone':          backbone_name,
        'cv_dice_mean':      float(np.mean(fold_dices)),
        'cv_dice_std':       float(np.std(fold_dices)),
        'fold_dices':        fold_dices,
        'ensemble_dice':     dice_ensemble,
        'ensemble_iou':      iou_ensemble,
        'cv_iou_mean':       float(np.mean(fold_ious)),
        'cv_iou_std':        float(np.std(fold_ious)),
        'ensemble_dice_std': float(dice_ens_std),
        'ensemble_iou_std':  float(iou_ens_std),
    })

    unet_ensembles[backbone_name] = ensemble

    torch.cuda.empty_cache()

# Summary
print(f"\n{'═'*60}")
print(f"  SEGMENTATION SUMMARY")
print(f"{'═'*60}")
print(f"  {'backbone':<20s} {'CV Dice (fold)':<22s} {'Ens Dice':<12s} {'Ens IoU':<10s}")
for r in seg_results_table:
    cv_str = f"{r['cv_dice_mean']:.4f} ± {r['cv_dice_std']:.4f}"
    print(f"  {r['backbone']:<20s} {cv_str:<22s} {r['ensemble_dice']:<12.4f} {r['ensemble_iou']:<10.4f}")
print(f"{'═'*60}")

In [ ]:
# IMAGE PREPROCESSING (one cache per pipeline variant)

print("Preprocessing images...")

full_dataset = datasets.ImageFolder(root=COMBINED_DATA_DIR, transform=None)
test_dataset_raw = datasets.ImageFolder(root=TEST_DIR, transform=None)

all_samples = {
    'train': full_dataset.samples,
    'test':  test_dataset_raw.samples,
}

image_caches = {}

# Variants with segmentation (one per U-Net backbone)
for backbone_name in BACKBONES_UNET:
    variant = f'seg_{backbone_name}'
    image_caches[variant] = {}

    preprocess_fn = FinalPreprocessing(
        width_factor=CURRENT_CROP_FACTOR,
        clip_limit=CURRENT_CLAHE_LIMIT,
        unet_model=unet_ensembles[backbone_name],
        unet_device=device,
        use_tta=True,
        blur_sigma=5.0,
        bg_attenuation=0.25,
        use_segmentation=True,
    )

    total = sum(len(s) for s in all_samples.values())
    counter = 0
    print(f"\n[{variant}] preprocessing {total} images…")

    for split, samples in all_samples.items():
        for img_path, _ in samples:
            img_pil = Image.open(img_path).convert('RGB')
            image_caches[variant][img_path] = preprocess_fn(img_pil)
            counter += 1
            if counter % 50 == 0 or counter == total:
                print(f"  {counter}/{total}", end='\r')
    print(f"\n  ✓ {variant}: {len(image_caches[variant])} images")

# Variant without segmentation (shared by all 3 architectures)
variant = 'no_seg'
image_caches[variant] = {}
preprocess_fn_noseg = FinalPreprocessing(
    width_factor=CURRENT_CROP_FACTOR,
    clip_limit=CURRENT_CLAHE_LIMIT,
    unet_model=None,
    unet_device=None,
    use_segmentation=False,
)

total = sum(len(s) for s in all_samples.values())
counter = 0
print(f"\n[{variant}] preprocessing {total} images…")
for split, samples in all_samples.items():
    for img_path, _ in samples:
        img_pil = Image.open(img_path).convert('RGB')
        image_caches[variant][img_path] = preprocess_fn_noseg(img_pil)
        counter += 1
        if counter % 50 == 0 or counter == total:
            print(f"  {counter}/{total}", end='\r')
print(f"\n  ✓ {variant}: {len(image_caches[variant])} images")

torch.cuda.empty_cache()

print(f"\n{'═'*60}")
print(f"Caches completed: {len(image_caches)} variants")
for variant, cache in image_caches.items():
    print(f"  {variant:25s} {len(cache)} images")
print(f"{'═'*60}")

In [ ]:
# CLASSIFICATION AUGMENTATION
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406] if USE_IMAGENET_NORM else [0., 0., 0.],
                                 std=[0.229, 0.224, 0.225] if USE_IMAGENET_NORM else [1., 1., 1.])

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomAffine(
        degrees=15,
        translate=(0.08, 0.08),
        scale=(0.90, 1.10),
        fill=0
    ),
    transforms.RandomPerspective(
        distortion_scale=0.1, p=0.3
    ),
    transforms.ColorJitter(
        brightness=0.20,
        contrast=0.20,
        saturation=0.15,
        hue=0.03
    ),
    transforms.ToTensor(),
    AddGaussianNoise(std=0.02, p=0.3),
    normalize,
    transforms.RandomErasing(
        p=0.25, scale=(0.02, 0.15),
        ratio=(0.3, 3.3), value=0
    ),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    normalize
])

In [ ]:
# CLASSIFICATION MODELS

def _make_head(in_features, hidden=256, dropout=0.4):
    # Same classification head for all architectures
    return nn.Sequential(
        nn.Flatten(1),
        nn.LayerNorm(in_features),
        nn.Linear(in_features, hidden),
        nn.GELU(),
        nn.Dropout(dropout),
        nn.Linear(hidden, 1),
    )

def get_classifier(arch_name, pretrained=True):
    if arch_name == 'densenet121':
        weights  = models.DenseNet121_Weights.DEFAULT if pretrained else None
        model    = models.densenet121(weights=weights)
        num_ftrs = model.classifier.in_features     # 1024
        model.classifier = _make_head(num_ftrs)

    elif arch_name == 'resnet18':
        weights  = models.ResNet18_Weights.DEFAULT if pretrained else None
        model    = models.resnet18(weights=weights)
        num_ftrs = model.fc.in_features             # 512
        model.fc = _make_head(num_ftrs)

    elif arch_name == 'efficientnet_v2_s':
        weights  = models.EfficientNet_V2_S_Weights.DEFAULT if pretrained else None
        model    = models.efficientnet_v2_s(weights=weights)
        num_ftrs = model.classifier[1].in_features  # 1280
        model.classifier = _make_head(num_ftrs)

    else:
        raise ValueError(f"Unknown architecture: {arch_name}")

    return model

def get_features_module(model, arch_name):
    """Returns the modules that make up the feature extractor (for freezing)."""
    if arch_name == 'resnet18':
        # ResNet has no single 'features' module
        return [model.conv1, model.bn1, model.layer1, model.layer2,
                model.layer3, model.layer4]
    else:
        return [model.features]

def get_head_params(model, arch_name):
    """Returns the parameters of the classification head."""
    if arch_name == 'resnet18':
        return list(model.fc.parameters())
    else:
        return list(model.classifier.parameters())

def set_features_grad(model, arch_name, requires_grad):
    """Enables/disables requires_grad of the feature extractor."""
    for module in get_features_module(model, arch_name):
        for p in module.parameters():
            p.requires_grad = requires_grad

def get_features_params(model, arch_name):
    """Returns the parameters of the feature extractor."""
    params = []
    for module in get_features_module(model, arch_name):
        params.extend(module.parameters())
    return params

In [ ]:
# CLASSIFICATION K-FOLD TRAINING

def train_pipeline(arch_name, variant, image_cache, full_dataset, k_folds=5):
    """
    Trains one pipeline: 3 seeds × 5 folds = 15 models.
    Saves the best checkpoint of each fold to disk; models with F1 >= MIN_F1_FOR_ENSEMBLE
    are selected for the ensemble.
    Returns: dictionary with cv_results, model_quality, optimal_thresholds, ensemble_threshold.
    """
    targets = full_dataset.targets

    print(f"\n{'═'*60}")
    print(f"  TRAINING PIPELINE: {arch_name}  |  variant: {variant}")
    print(f"{'═'*60}")

    cv_results = []
    optimal_thresholds = []
    model_quality = {}

    for seed_idx, seed in enumerate(SEEDS):
        print(f"\n{'='*50}")
        print(f"  SEED {seed_idx+1}/{len(SEEDS)} — random_state={seed}")
        print(f"{'='*50}")

        skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=seed)

        for fold, (train_idx, val_idx) in enumerate(
            skf.split(np.zeros(len(targets)), targets)
        ):
            print(f"\n  [{arch_name}|{variant}] SEED {seed} — FOLD {fold + 1}/{k_folds}")

            train_sub = Subset(full_dataset, train_idx)
            val_sub   = Subset(full_dataset, val_idx)

            train_data = DatasetWrapper(train_sub, transform=train_transforms,
                                        image_cache=image_cache)
            val_data   = DatasetWrapper(val_sub,   transform=val_transforms,
                                        image_cache=image_cache)

            train_loader = DataLoader(
                train_data, batch_size=32, shuffle=True,
                num_workers=4, pin_memory=True, persistent_workers=True
            )
            val_loader = DataLoader(
                val_data, batch_size=32, shuffle=False,
                num_workers=4, pin_memory=True, persistent_workers=True
            )

            criterion = FocalBCEWithLogitsLoss(gamma=2.0, pos_weight=pos_weight_tensor)

            model_prefix = f'{arch_name}_{variant}_s{seed}_fold_{fold+1}'

            # Instantiate the classifier (same head for all architectures)
            model = get_classifier(arch_name, pretrained=True).to(device)

            # Freeze the feature extractor for the first epochs
            set_features_grad(model, arch_name, requires_grad=False)

            optimizer = optim.AdamW(
                get_head_params(model, arch_name),
                lr=HEAD_LR, weight_decay=0.03
            )
            scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='max', factor=0.5, patience=4
            )
            scaler = GradScaler()

            NUM_EPOCHS  = 50
            best_f1_opt = 0.0
            best_fold_metrics = {
                'f1': 0, 'recall': 0, 'specificity': 0,
                'precision': 0, 'balanced_acc': 0, 'threshold': 0.5
            }
            es_counter = 0

            for epoch in range(NUM_EPOCHS):

                # Unfreeze the feature extractor at the end of the initial freeze
                if epoch == FREEZE_EPOCHS:
                    set_features_grad(model, arch_name, requires_grad=True)
                    optimizer = optim.AdamW([
                        {'params': get_features_params(model, arch_name),
                         'lr': BACKBONE_LR},
                        {'params': get_head_params(model, arch_name),
                         'lr': optimizer.param_groups[0]['lr']},
                    ], weight_decay=0.03)
                    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                        optimizer, mode='max', factor=0.5, patience=4
                    )

                # Training
                model.train()
                for images, labels in train_loader:
                    images = images.to(device, non_blocking=True)
                    labels = labels.float().unsqueeze(1).to(device, non_blocking=True)
                    labels_smooth = smooth_labels(labels, LABEL_SMOOTH)

                    optimizer.zero_grad()
                    with autocast(device_type='cuda'):
                        outputs = model(images)
                        loss = criterion(outputs, labels_smooth)

                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    scaler.step(optimizer)
                    scaler.update()

                # Validation
                model.eval()
                all_labels_list = []
                all_probs_list  = []
                with torch.no_grad():
                    for images, labels_batch in val_loader:
                        images = images.to(device, non_blocking=True)
                        labels_batch = labels_batch.float().unsqueeze(1).to(device)
                        logits_tta = classify_with_tta_light(model, images, device)
                        probs = torch.sigmoid(logits_tta)
                        all_labels_list.extend(labels_batch.cpu().numpy().flatten())
                        all_probs_list.extend(probs.cpu().numpy().flatten())

                all_labels_np = np.array(all_labels_list)
                all_probs_np  = np.array(all_probs_list)

                # Optimal threshold search
                if len(np.unique(all_labels_np)) > 1:
                    fpr, tpr, thresholds_roc = roc_curve(all_labels_np, all_probs_np)
                    best_target_f1 = 0.0
                    best_target_thresh = 0.5
                    for t in np.arange(0.30, 0.85, 0.01):
                        p_opt = (all_probs_np >= t).astype(float)
                        tn_t, fp_t, fn_t, tp_t = confusion_matrix(
                            all_labels_np, p_opt, labels=[0,1]
                        ).ravel()
                        spec_t = tn_t / (tn_t + fp_t) if (tn_t + fp_t) > 0 else 0.0
                        sens_t = tp_t / (tp_t + fn_t) if (tp_t + fn_t) > 0 else 0.0

                        if spec_t >= 0.85 and sens_t >= 0.70:
                            f1_t = f1_score(all_labels_np, p_opt, zero_division=0)
                            if f1_t > best_target_f1:
                                best_target_f1 = f1_t
                                best_target_thresh = t

                    if best_target_f1 == 0.0:
                        j_scores = tpr - fpr
                        best_target_thresh = thresholds_roc[np.argmax(j_scores)]

                    optimal_thresh = best_target_thresh
                else:
                    optimal_thresh = 0.5

                preds_opt = (all_probs_np >= optimal_thresh).astype(float)
                f1_opt = f1_score(all_labels_np, preds_opt, zero_division=0)
                scheduler.step(f1_opt)

                tn_o, fp_o, fn_o, tp_o = confusion_matrix(
                    all_labels_np, preds_opt, labels=[0,1]
                ).ravel()
                spec_opt = tn_o / (tn_o + fp_o) if (tn_o + fp_o) > 0 else 0.0
                rec_opt  = recall_score(all_labels_np, preds_opt, zero_division=0)

                if f1_opt > best_f1_opt:
                    best_f1_opt = f1_opt
                    best_fold_metrics = {
                        'f1': f1_opt, 'recall': rec_opt, 'specificity': spec_opt,
                        'precision':    precision_score(all_labels_np, preds_opt, zero_division=0),
                        'balanced_acc': balanced_accuracy_score(all_labels_np, preds_opt),
                        'threshold': optimal_thresh
                    }

                    # Save the best checkpoint of the fold (needed if the top-3 fallback includes it)
                    torch.save(model.state_dict(),
                               f'best_cls_{model_prefix}.pth')

                    if f1_opt >= MIN_F1_FOR_ENSEMBLE:
                        model_quality[model_prefix] = {
                            'f1': f1_opt,
                            'arch_name': arch_name,
                            'prefix': model_prefix,
                        }
                    es_counter = 0
                else:
                    es_counter += 1

                if epoch % 10 == 0 or es_counter >= ES_PATIENCE:
                    print(f"    Ep {epoch+1:02d} | F1: {f1_opt:.3f} at {optimal_thresh:.2f} | "
                          f"Sens: {rec_opt:.2f} | Spec: {spec_opt:.2f} | ES: {es_counter}/{ES_PATIENCE}")

                if es_counter >= ES_PATIENCE:
                    print(f"    Early stopping at epoch {epoch+1}")
                    break

            cv_results.append({
                **best_fold_metrics,
                'seed': seed,
                'fold': fold + 1,
                'prefix': model_prefix,
            })
            optimal_thresholds.append(best_fold_metrics['threshold'])

            quality_tag = "✓" if best_f1_opt >= MIN_F1_FOR_ENSEMBLE else "✗"
            print(f"  {quality_tag} [{arch_name}|{variant}] s{seed} fold_{fold+1}: "
                  f"F1={best_fold_metrics['f1']:.3f} "
                  f"Sens={best_fold_metrics['recall']:.2f} "
                  f"Spec={best_fold_metrics['specificity']:.2f}")

            # Free memory between folds
            del model, optimizer, scheduler, scaler
            torch.cuda.empty_cache()

    # Top-3 fallback: if fewer than 3 models passed the threshold,
    # take the top-3 by F1 anyway (among ALL 15 folds)
    if len(model_quality) < MIN_MODELS_FALLBACK:
        print(f"\n  Only {len(model_quality)} models reached F1 > {MIN_F1_FOR_ENSEMBLE}")
        print(f"     Fallback: using top-{MIN_MODELS_FALLBACK} by F1")
        # Sort all folds by decreasing F1
        sorted_results = sorted(cv_results, key=lambda r: r['f1'], reverse=True)
        model_quality = {}
        for r in sorted_results[:MIN_MODELS_FALLBACK]:
            model_quality[r['prefix']] = {
                'f1': r['f1'],
                'arch_name': arch_name,
                'prefix': r['prefix'],
            }

    print(f"\n  Models in the ensemble: {len(model_quality)}")
    for k, v in model_quality.items():
        print(f"    {k}: F1={v['f1']:.3f}")

    ensemble_threshold = float(np.median(optimal_thresholds))

    return {
        'arch_name': arch_name,
        'variant': variant,
        'cv_results': cv_results,
        'model_quality': model_quality,
        'optimal_thresholds': optimal_thresholds,
        'ensemble_threshold': ensemble_threshold,
    }


# Loop over the pipelines
pipeline_results = {}

for arch_name in ARCHITECTURES_CLS:
    # Pipeline A: with segmentation
    unet_backbone = {
        'efficientnet_v2_s': 'efficientnet-b0',
        'resnet18':          'resnet18',
        'densenet121':       'densenet121',
    }[arch_name]
    variant_seg = f'seg_{unet_backbone}'
    pipeline_results[(arch_name, variant_seg)] = train_pipeline(
        arch_name, variant_seg, image_caches[variant_seg], full_dataset
    )

    # Pipeline B: without segmentation
    pipeline_results[(arch_name, 'no_seg')] = train_pipeline(
        arch_name, 'no_seg', image_caches['no_seg'], full_dataset
    )

print(f"\n\n{'═'*60}")
print(f"  TRAINING COMPLETED — {len(pipeline_results)} pipelines")
print(f"{'═'*60}")
for (arch, variant), res in pipeline_results.items():
    n_ens = len(res['model_quality'])
    print(f"  {arch:22s} | {variant:25s} | {n_ens} ensemble models")
print(f"{'═'*60}")

In [ ]:
# CLASSIFICATION K-FOLD RESULTS

print(f"\n{'═'*80}")
print(f"  CLASSIFICATION K-FOLD RESULTS — summary per pipeline")
print(f"{'═'*80}")
print(f"  {'Pipeline':<45s}  {'Bal-Acc':>13s}  {'F1':>13s}  {'Sens':>13s}  {'Spec':>13s}  {'Prec':>13s}  {'Thr':>6s}  {'Ens':>4s}")
print(f"  {'─'*134}")

for (arch_name, variant), res in pipeline_results.items():
    cv_results         = res['cv_results']
    optimal_thresholds = res['optimal_thresholds']
    model_quality      = res['model_quality']

    f1_mean   = np.mean([r['f1']          for r in cv_results])
    f1_std    = np.std([r['f1']           for r in cv_results])
    rec_mean  = np.mean([r['recall']      for r in cv_results])
    rec_std   = np.std([r['recall']       for r in cv_results])
    spec_mean = np.mean([r['specificity'] for r in cv_results])
    spec_std  = np.std([r['specificity']  for r in cv_results])
    bal_mean  = np.mean([r['balanced_acc'] for r in cv_results])
    bal_std   = np.std([r['balanced_acc']  for r in cv_results])
    prec_mean = np.mean([r['precision']    for r in cv_results])
    prec_std  = np.std([r['precision']     for r in cv_results])
    thr_med   = np.median(optimal_thresholds)

    label = f"{arch_name} | {variant}"
    print(f"  {label:<45s}  "
          f"{bal_mean:.3f} ± {bal_std:.3f}  "
          f"{f1_mean:.3f} ± {f1_std:.3f}  "
          f"{rec_mean:.3f} ± {rec_std:.3f}  "
          f"{spec_mean:.3f} ± {spec_std:.3f}  "
          f"{prec_mean:.3f} ± {prec_std:.3f}  "
          f"{thr_med:>6.3f}  "
          f"{len(model_quality):>4d}")

In [ ]:
# TEST SET EVALUATION

print("\n--- TEST SET EVALUATION ---")

# Test set images are already preprocessed in the caches
test_true_labels = np.array([label for _, label in test_dataset_raw.samples])
print(f"Test set: {len(test_true_labels)} images "
      f"(Negative: {(test_true_labels==0).sum()}, Positive: {(test_true_labels==1).sum()})")

# Class folders: NO -> 0 (negative), SI -> 1 (positive)
target_names = ['Negative', 'Positive']

# Evaluation of a single pipeline
def evaluate_pipeline_on_test(arch_name, variant, pipeline_info, image_cache):
    """
    Loads the quality-gated ensemble models, runs TTA inference on the test set,
    aggregates with an F1-weighted mean, applies the optimized median threshold,
    returns a dict with the complete metrics.
    """
    print(f"\n{'─'*60}")
    print(f"  {arch_name:22s} | {variant:25s}")
    print(f"{'─'*60}")

    model_quality      = pipeline_info['model_quality']
    ensemble_threshold = pipeline_info['ensemble_threshold']

    if len(model_quality) == 0:
        print("  ✗ No models in the ensemble. Skip.")
        return None

    # DataLoader with the pipeline-specific cache
    test_data = DatasetWrapper(
        Subset(test_dataset_raw, list(range(len(test_dataset_raw)))),
        transform=val_transforms, image_cache=image_cache
    )
    test_loader = DataLoader(test_data, batch_size=32, shuffle=False, num_workers=0)

    # TTA inference with each ensemble model
    all_fold_probs   = []
    all_fold_weights = []

    for model_key, model_info in model_quality.items():
        f1_weight = model_info['f1']
        path_base = f'best_cls_{model_key}.pth'

        try:
            m = get_classifier(arch_name, pretrained=False).to(device)
            m.load_state_dict(torch.load(path_base, map_location=device,
                                         weights_only=True))
            m.eval()

            fold_probs = []
            with torch.no_grad():
                for images, _ in test_loader:
                    images = images.to(device, non_blocking=True)
                    logits_tta = classify_with_tta(m, images, device)
                    probs = torch.sigmoid(logits_tta).cpu().numpy().flatten()
                    fold_probs.extend(probs)

            all_fold_probs.append(fold_probs)
            all_fold_weights.append(f1_weight)
            print(f"    ✓ {model_key}  (F1={f1_weight:.3f})")

            del m
            torch.cuda.empty_cache()

        except FileNotFoundError:
            print(f" ✗ {path_base} not found")

    if len(all_fold_probs) == 0:
        print("  ✗ No checkpoint loaded. Skip.")
        return None

    # Aggregation: mean weighted by validation F1
    all_fold_probs_np = np.array(all_fold_probs)
    all_fold_weights_np = np.array(all_fold_weights)
    ensemble_probs = np.average(all_fold_probs_np, axis=0, weights=all_fold_weights_np)

    # Apply the optimized threshold
    final_preds = (ensemble_probs >= ensemble_threshold).astype(int)

    # Metrics
    cm = confusion_matrix(test_true_labels, final_preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    accuracy    = (tp + tn) / (tp + tn + fp + fn)
    precision_Positive = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall_Positive    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    precision_Negative = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    recall_Negative    = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1_Positive = f1_score(test_true_labels, final_preds, pos_label=1, zero_division=0)
    f1_Negative = f1_score(test_true_labels, final_preds, pos_label=0, zero_division=0)
    balanced_acc = balanced_accuracy_score(test_true_labels, final_preds)

    print(f"\n  Ensemble: {len(all_fold_probs)} models | Threshold: {ensemble_threshold:.3f}")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Recall Positive:    {recall_Positive:.4f}  |  Precision Positive: {precision_Positive:.4f}  |  F1 Positive: {f1_Positive:.4f}")
    print(f"  Recall Negative:    {recall_Negative:.4f}  |  Precision Negative: {precision_Negative:.4f}  |  F1 Negative: {f1_Negative:.4f}")
    print(f"  Balanced Acc: {balanced_acc:.4f}")

    # Per-image ensemble predictions on the test set (for bootstrap confidence intervals and McNemar's test)
    pd.DataFrame({
        'run': 'full', 'arch': arch_name, 'variant': variant,
        'file': [os.path.basename(p) for p, _ in test_dataset_raw.samples],
        'label': test_true_labels, 'prob': ensemble_probs, 'pred': final_preds,
    }).to_csv(f'pred_test_{arch_name}_{variant}.csv', index=False)

    return {
        'arch_name':          arch_name,
        'variant':            variant,
        'n_ensemble_models':  len(all_fold_probs),
        'threshold':          ensemble_threshold,
        'accuracy':           accuracy,
        'balanced_accuracy':  balanced_acc,
        'precision_Positive':       precision_Positive,
        'recall_Positive':          recall_Positive,
        'f1_Positive':              f1_Positive,
        'precision_Negative':       precision_Negative,
        'recall_Negative':          recall_Negative,
        'f1_Negative':              f1_Negative,
        'confusion_matrix':   cm.tolist(),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }


# Loop over the pipelines
cls_results_table = []

for (arch_name, variant), pipeline_info in pipeline_results.items():
    image_cache = image_caches[variant]
    result = evaluate_pipeline_on_test(arch_name, variant, pipeline_info, image_cache)
    if result is not None:
        cls_results_table.append(result)

print(f"\n\n{'═'*60}")
print(f"  EVALUATION COMPLETED — {len(cls_results_table)} pipelines evaluated")
print(f"{'═'*60}")

In [ ]:
# TEST SET: MEAN ± STD ACROSS INDIVIDUAL MODELS (seeds × folds)

per_model_rows = []

for (arch_name, variant), res in pipeline_results.items():
    test_data = DatasetWrapper(Subset(test_dataset_raw, list(range(len(test_dataset_raw)))),
                               transform=val_transforms, image_cache=image_caches[variant])
    test_loader = DataLoader(test_data, batch_size=32, shuffle=False, num_workers=0)

    for r in res['cv_results']:
        m = get_classifier(arch_name, pretrained=False).to(device)
        m.load_state_dict(torch.load(f"best_cls_{r['prefix']}.pth", map_location=device, weights_only=True))
        m.eval()
        probs = []
        with torch.no_grad():
            for images, _ in test_loader:
                probs.extend(torch.sigmoid(classify_with_tta(m, images, device)).float().cpu().numpy().flatten())
        preds = (np.array(probs) >= r['threshold']).astype(int)
        tn, fp, fn, tp = confusion_matrix(test_true_labels, preds, labels=[0, 1]).ravel()
        per_model_rows.append({
            'architecture': arch_name, 'pipeline': variant, 'seed': r['seed'], 'fold': r['fold'],
            'balanced_acc': balanced_accuracy_score(test_true_labels, preds),
            'f1':           f1_score(test_true_labels, preds, zero_division=0),
            'sensitivity':  tp / (tp + fn) if (tp + fn) > 0 else 0.0,
            'specificity':  tn / (tn + fp) if (tn + fp) > 0 else 0.0,
            'precision':    tp / (tp + fp) if (tp + fp) > 0 else 0.0,
        })
        del m
        torch.cuda.empty_cache()

df_per_model = pd.DataFrame(per_model_rows)
metrics = ['balanced_acc', 'f1', 'sensitivity', 'specificity', 'precision']
summary = df_per_model.groupby(['architecture', 'pipeline'])[metrics].agg(
    lambda x: f"{x.mean():.3f} ± {x.std(ddof=0):.3f}")

print("\nTEST SET — mean ± standard deviation across individual models (seeds x folds)\n")
print(summary.to_string())
df_per_model.to_csv('test_per_model.csv', index=False)
summary.to_csv('test_mean_std.csv')

In [ ]:
# FINAL TABLES

# TABLE 1: Segmentation
print(f"\n{'═'*80}")
print(f"  TABLE 1 — SEGMENTATION")
print(f"{'═'*80}")

# Mean ± standard deviation for both Dice and IoU
print(f"{'Backbone':<18}{'CV Dice (folds)':<20}{'CV IoU (folds)':<20}{'Test Dice (images)':<22}{'Test IoU (images)':<22}")
print('─' * 102)
for r in seg_results_table:
    print(f"{r['backbone']:<18}"
          f"{r['cv_dice_mean']:.4f} ± {r['cv_dice_std']:.4f}     "
          f"{r['cv_iou_mean']:.4f} ± {r['cv_iou_std']:.4f}     "
          f"{r['ensemble_dice']:.4f} ± {r['ensemble_dice_std']:.4f}       "
          f"{r['ensemble_iou']:.4f} ± {r['ensemble_iou_std']:.4f}")
print('─' * 102)

# TABLE 2: Classification
def arch_label(arch_name):
    return {
        'densenet121':       'DenseNet-121',
        'resnet18':          'ResNet-18',
        'efficientnet_v2_s': 'EfficientNet-V2-S',
    }.get(arch_name, arch_name)

def variant_label(variant):
    if variant == 'no_seg':
        return 'Without segmentation'
    return f'With U-Net ({variant.replace("seg_", "")})'

# Sort by architecture, then by variant (without segmentation first)
sorted_results = sorted(
    cls_results_table,
    key=lambda x: (x['arch_name'], 0 if x['variant'] == 'no_seg' else 1)
)

# Overall metrics
print(f"\n{'═'*95}")
print(f"  TABLE 2 — CLASSIFICATION ON TEST SET  (overall metrics)")
print(f"{'═'*95}")

W_ARCH = 22; W_PIPE = 30; W_ENS = 10; W_THR = 12; W_ACC = 12; W_BAL = 14

header = (
    f"{'Architecture':<{W_ARCH}}"
    f"{' ':<{W_PIPE}}"
    f"{'Models':<{W_ENS}}"
    f"{'Threshold':<{W_THR}}"
    f"{'Accuracy':<{W_ACC}}"
    f"{'Balanced Acc':<{W_BAL}}"
)
print(header)
print('─' * (W_ARCH + W_PIPE + W_ENS + W_THR + W_ACC + W_BAL))

for r in sorted_results:
    print(
        f"{arch_label(r['arch_name']):<{W_ARCH}}"
        f"{variant_label(r['variant']):<{W_PIPE}}"
        f"{r['n_ensemble_models']:<{W_ENS}d}"
        f"{r['threshold']:<{W_THR}.3f}"
        f"{r['accuracy']:<{W_ACC}.4f}"
        f"{r['balanced_accuracy']:<{W_BAL}.4f}"
    )
print('─' * (W_ARCH + W_PIPE + W_ENS + W_THR + W_ACC + W_BAL))

# Per-class metrics
print(f"\n{'═'*110}")
print(f"  TABLE 2 — CLASSIFICATION ON TEST SET  (per-class metrics)")
print(f"{'═'*110}")

W_PREC = 12; W_REC = 12; W_F1 = 12

header = (
    f"{'Architecture':<{W_ARCH}}"
    f"{' ':<{W_PIPE}}"
    f"{'Prec (Pos)':<{W_PREC}}"
    f"{'Rec (Pos)':<{W_REC}}"
    f"{'F1 (Pos)':<{W_F1}}"
    f"{'Prec (Neg)':<{W_PREC}}"
    f"{'Rec (Neg)':<{W_REC}}"
    f"{'F1 (Neg)':<{W_F1}}"
)
print(header)
print('─' * (W_ARCH + W_PIPE + 3*W_PREC + 3*W_F1))

for r in sorted_results:
    print(
        f"{arch_label(r['arch_name']):<{W_ARCH}}"
        f"{variant_label(r['variant']):<{W_PIPE}}"
        f"{r['precision_Positive']:<{W_PREC}.4f}"
        f"{r['recall_Positive']:<{W_REC}.4f}"
        f"{r['f1_Positive']:<{W_F1}.4f}"
        f"{r['precision_Negative']:<{W_PREC}.4f}"
        f"{r['recall_Negative']:<{W_REC}.4f}"
        f"{r['f1_Negative']:<{W_F1}.4f}"
    )
print('─' * (W_ARCH + W_PIPE + 3*W_PREC + 3*W_F1))

# SUMMARY

print(f"\n\n{'═'*90}")
print(f"  SUMMARY")
print(f"{'═'*90}")

# Segmentation
print(f"\nSEGMENTATION (ensemble metrics on the test set):\n")

W_BACK = 22; W_DICE = 16; W_IOU = 16

header = (
    f"  {'Backbone':<{W_BACK}}"
    f"{'Dice':<{W_DICE}}"
    f"{'IoU':<{W_IOU}}"
)
print(header)
print('  ' + '─' * (W_BACK + W_DICE + W_IOU))

for r in seg_results_table:
    print(
        f"  {r['backbone']:<{W_BACK}}"
        f"{r['ensemble_dice']:<{W_DICE}.4f}"
        f"{r['ensemble_iou']:<{W_IOU}.4f}"
    )


# Classification
print(f"\n\nCLASSIFICATION (test set):\n")

W_ARCH = 22; W_PIPE = 32; W_MET = 12

header = (
    f"  {'Architecture':<{W_ARCH}}"
    f"{' ':<{W_PIPE}}"
    f"{'Acc':<{W_MET}}"
    f"{'Rec-Positive':<{W_MET}}"
    f"{'Prec-Positive':<{W_MET}}"
    f"{'F1-Positive':<{W_MET}}"
)
print(header)
print('  ' + '─' * (W_ARCH + W_PIPE + 4 * W_MET))

for r in sorted(cls_results_table, key=lambda x: (x['arch_name'], x['variant'])):
    print(
        f"  {arch_label(r['arch_name']):<{W_ARCH}}"
        f"{variant_label(r['variant']):<{W_PIPE}}"
        f"{r['accuracy']:<{W_MET}.4f}"
        f"{r['recall_Positive']:<{W_MET}.4f}"
        f"{r['precision_Positive']:<{W_MET}.4f}"
        f"{r['f1_Positive']:<{W_MET}.4f}"
    )

print("\n\n")

# Confusion Matrix

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for ax, r in zip(axes, sorted_results):
    cm = np.array(r['confusion_matrix'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=target_names, yticklabels=target_names, cbar=False)
    title = f"{arch_label(r['arch_name'])}\n{variant_label(r['variant'])}\n"
    ax.set_title(title, fontsize=10)
    ax.set_ylabel('Ground truth')
    ax.set_xlabel('Prediction')

for ax in axes[len(sorted_results):]:
    ax.axis('off')

plt.suptitle('Confusion matrices — 6 pipelines', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# CLASSIFICATION PERFORMANCE PLOT

# 1. Data preparation (sorted_results comes from the previous cell)
metrics_data = []
for r in sorted_results:
    arch = arch_label(r['arch_name'])
    suffix = " + U-Net" if r['variant'] != 'no_seg' else ""
    model_name = f"{arch}{suffix}"

    # Four metrics for each model
    metrics_data.append({'Model': model_name, 'Metric': 'Balanced Accuracy', 'Score': r['balanced_accuracy']})
    metrics_data.append({'Model': model_name, 'Metric': 'Sensitivity (Recall Positive)', 'Score': r['recall_Positive']})
    metrics_data.append({'Model': model_name, 'Metric': 'Specificity (Recall Negative)', 'Score': r['recall_Negative']})
    metrics_data.append({'Model': model_name, 'Metric': 'F1-Score (Positive)', 'Score': r['f1_Positive']})

df_metrics = pd.DataFrame(metrics_data)

# 2. Plot style (scientific style)
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'sans-serif'
fig, ax = plt.subplots(figsize=(15, 8))

# Colorblind-friendly palette (4 colors)
paper_palette = {
    'Balanced Accuracy': '#CC79A7',
    'Sensitivity (Recall Positive)': '#0072B2',
    'Specificity (Recall Negative)': '#D55E00',
    'F1-Score (Positive)': '#009E73'
}

# 3. Grouped bar chart
plot = sns.barplot(
    data=df_metrics,
    x='Model',
    y='Score',
    hue='Metric',
    palette=paper_palette,
    edgecolor='black',
    linewidth=0.8,
    alpha=0.9
)

# 4. Title and axis labels
ax.set_title('Classification Performance Comparison\nBalanced Accuracy, Sensitivity, Specificity, and F1-Score',
             fontsize=16, fontweight='bold', pad=30)

ax.set_xlabel('Model Configuration', fontsize=12, fontweight='medium', labelpad=15)
ax.set_ylabel('Metric Score (0.0 - 1.0)', fontsize=12, fontweight='medium', labelpad=15)
ax.set_ylim(0, 1.15)  # extra space for the value labels

# Legend placed below the plot so that it does not cover the bars
ax.legend(title='Key Performance Indicators', frameon=True, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=4)

sns.despine()

# 5. Value labels above the bars
for p in plot.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{height:.2f}',
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom',
                    xytext=(0, 5),
                    textcoords='offset points',
                    fontsize=8.5, fontweight='bold')

# Rotate the x labels for readability
plt.xticks(rotation=15, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# SEGMENTATION EXAMPLES PLOT

def visualize_segmentation_examples(ensembles, dataset, device, n_examples=3):
    """
    Shows the comparison between the ground truth and the predictions of the U-Net backbones.
    """
    # Randomly select n_examples samples
    indices = random.sample(range(len(dataset)), n_examples)
    backbones = list(ensembles.keys())
    n_cols = 2 + len(backbones)  # image + GT + one column per backbone

    fig, axes = plt.subplots(n_examples, n_cols, figsize=(4 * n_cols, 4 * n_examples))

    # Column titles
    titles = ['Original Image', 'Ground Truth'] + [f'Pred: {b}' for b in backbones]

    for row_idx, sample_idx in enumerate(indices):
        # Get the sample from the dataset
        image_tensor, mask_tensor = dataset[sample_idx]

        # Convert the image to (H, W, C) for display
        img_display = image_tensor.permute(1, 2, 0).cpu().numpy()
        # Rescale the normalized image to [0, 1]
        img_display = (img_display - img_display.min()) / (img_display.max() - img_display.min() + 1e-8)

        mask_display = mask_tensor.squeeze().cpu().numpy()

        # Column 0: input image
        axes[row_idx, 0].imshow(img_display)
        axes[row_idx, 0].set_axis_off()
        if row_idx == 0: axes[row_idx, 0].set_title(titles[0], fontweight='bold')

        # Column 1: ground truth
        axes[row_idx, 1].imshow(mask_display, cmap='gray')
        axes[row_idx, 1].set_axis_off()
        if row_idx == 0: axes[row_idx, 1].set_title(titles[1], fontweight='bold')

        # Remaining columns: predictions of each backbone
        for col_idx, backbone_name in enumerate(backbones):
            ensemble = ensembles[backbone_name]

            # Prediction
            with torch.no_grad():
                # Add the batch dimension [1, C, H, W]
                input_tensor = image_tensor.unsqueeze(0).to(device)
                pred_prob = ensemble(input_tensor)
                pred_mask = (torch.sigmoid(pred_prob) > 0.5).squeeze().cpu().numpy()

            ax = axes[row_idx, col_idx + 2]
            ax.imshow(pred_mask, cmap='viridis')  # different colormap to distinguish predictions from the GT
            ax.set_axis_off()
            if row_idx == 0:
                ax.set_title(titles[col_idx + 2], fontweight='bold')

    plt.suptitle("Segmentation Results Comparison across Different UNet Backbones",
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# Run the visualization on the test masks (dataset of full_seg_loader)
visualize_segmentation_examples(
    unet_ensembles,
    full_seg_loader.dataset,
    device,
    n_examples=5
)